# Rule-Based PM 스케줄러 시뮬레이션

## PM 결정 로직 요약

| 케이스 | 조건 | 결정 | ΔC_max |
|--------|------|------|--------|
| A (IDLE_FIT) | idle ≥ pm_duration | PM을 유휴 구간에 삽입 | 0 |
| B (ADVANCE)  | ΔC_max(adv) ≤ ΔC_max(del) | PM → 작업 순서 | max(0, pm_dur - idle) |
| C (DELAY)    | ΔC_max(adv) > ΔC_max(del) | 작업 → PM 순서 | repair_time × Λ(op 구간) |

## 환경 설정

In [1]:
import os
import simpy
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

SIMUL_TIME     = int(os.getenv('SIMUL_TIME', 100))
BASE_DATA_PATH = os.getenv('BASE_DATA_PATH', 'data')

# PM 판단 임계치 ε (튜닝 파라미터)
# 낮을수록 더 자주 PM 수행 → 고장 위험 감소, 가동 중단 증가
PM_EPSILON = float(os.getenv('PM_EPSILON', 0.4))

print(f'시뮬레이션 시간 : {SIMUL_TIME}')
print(f'데이터 경로     : {BASE_DATA_PATH}')
print(f'PM 임계치 (ε)   : {PM_EPSILON}')

시뮬레이션 시간 : 100
데이터 경로     : data
PM 임계치 (ε)   : 0.4


## 모듈 import

In [2]:
from utils import DataLoader, EventLogger
from utils.visualizer import create_gantt_chart
from simulation import Scheduler, Machine, Job

# RuleBasedScheduler: Scheduler를 상속하며 PM advance/delay 로직 포함
from algorithms.rule_based.scheduler import RuleBasedScheduler, PMDecision

## 데이터 로드

In [3]:
data_loader = DataLoader(BASE_DATA_PATH)
data = data_loader.load_all_data()

print('=' * 60)
print('데이터 개요')
print('=' * 60)
print(f"Jobs             : {len(data['jobs'])} 개")
print(f"Operations       : {len(data['operations'])} 개")
print(f"Machines         : {len(data['machines'])} 개")
print(f"Machine Failures : {len(data['machine_failure'])} 개")
print(f"Setup Times      : {len(data['setup_times'])} 개")
print(f"Op-Machine Map   : {len(data['operation_machine_map'])} 개")

데이터 개요
Jobs             : 10 개
Operations       : 35 개
Machines         : 8 개
Machine Failures : 8 개
Setup Times      : 12 개
Op-Machine Map   : 95 개


## 시뮬레이션 실행

In [4]:
from utils import EventLogger
env = simpy.Environment()
event_logger = EventLogger(env)

# ── 기존 Scheduler 대신 RuleBasedScheduler 사용 ──────────────────────
scheduler = RuleBasedScheduler(
    env=env,
    machine_df=data['machines'],
    operations_df=data['operations'],
    machine_failure_df=data['machine_failure'],
    setup_times_df=data['setup_times'],
    op_machine_df=data['operation_machine_map'],
    event_logger=event_logger,
    pm_epsilon=PM_EPSILON,
)

In [5]:
jobs = []
for _, job_info in data['jobs'].iterrows():
    # 해당 작업의 operation 정보 가져오기
    job_operations = data['operations'].loc[
        data['operations']['job_id'] == job_info['job_id'],
        ['op_id', 'op_seq', 'qtime']
    ].sort_values('op_seq')
    
    # Job 인스턴스 생성
    job = Job(
        env=env,
        job_info=job_info.to_dict(),
        op_info=job_operations,
        scheduler=scheduler,
        event_logger=event_logger
    )
    jobs.append(job)

In [6]:
from utils import create_gantt_chart
from IPython.display import display
import plotly.io as pio

env.run(until=env.all_of([job.process for job in jobs]))   # generator → list

fig = create_gantt_chart(
    logs=event_logger.logs,
    max_time=env.now,
    title=f"반도체 공정 간트 차트 (Simulation Time: {round(env.now, 3)})"
)
display(fig)   # fig.show() 대신 display() 사용

39	Machine M3 needs PM (Λ=0.4602 ≥ ε=0.4)
39	Machine M3 → PM Decision: ΔC_max(Adv)=5.0000  ΔC_max(Del)=2.3112
39	Machine M3 → DELAY: PM after job (job_id=J9, op_seq=1)
41	Machine M2 needs PM (Λ=0.6253 ≥ ε=0.4)
41	Machine M2 → PM Decision: ΔC_max(Adv)=5.0000  ΔC_max(Del)=3.1075
41	Machine M2 → DELAY: PM after job (job_id=J1, op_seq=3)
43	Machine M1 needs PM (Λ=0.6773 ≥ ε=0.4)
43	Machine M1 → PM Decision: ΔC_max(Adv)=5.0000  ΔC_max(Del)=6.3000
43	Machine M1 → ADVANCE: PM before job
43	Machine M1 PM started (duration=5)
48	Machine M1 PM finished (last_repair_time → 48.00)
53	Machine M3 PM started (duration=5)
54	Machine M4 needs PM (Λ=0.5994 ≥ ε=0.4)
54	Machine M4 → PM Decision: ΔC_max(Adv)=6.0000  ΔC_max(Del)=5.5462
54	Machine M4 → DELAY: PM after job (job_id=J6, op_seq=2)
57	Machine M2 PM started (duration=5)
57	Machine M7 needs PM (Λ=0.4389 ≥ ε=0.4)
57	Machine M7 → PM Decision: IDLE_FIT (idle=57.00 ≥ pm_dur=4)
57	Machine M7 → IDLE_FIT: PM before job
57	Machine M7 PM started (duration=4

## PM 수행 이력 분석

In [12]:
pm_df = scheduler.get_pm_summary()

print(f'총 PM 수행 횟수: {len(pm_df)}')

if not pm_df.empty:
    print('\n[PM 결정 유형별 집계]')
    print(pm_df['decision'].value_counts())

    print('\n[PM 수행 이력]')
    pm_df['pm_duration_actual'] = pm_df['pm_end'] - pm_df['pm_start']
    display(pm_df)

총 PM 수행 횟수: 14

[PM 결정 유형별 집계]
decision
IDLE_FIT    9
ADVANCE     3
DELAY       2
Name: count, dtype: int64

[PM 수행 이력]


,machine_id,decision,pm_start,pm_end,job_id,op_seq,pm_duration_actual
0,M3,ADVANCE,39.000000,44.000000,J8,1.0,5.0
1,M1,IDLE_FIT,54.000000,59.000000,J1,3.0,5.0
2,M4,ADVANCE,54.000000,60.000000,J5,2.0,6.0
3,M2,DELAY,57.000000,62.000000,NaN,NaN,5.0
4,M8,IDLE_FIT,63.000000,67.000000,J6,3.0,4.0
5,M5,DELAY,63.000000,69.000000,NaN,NaN,6.0
6,M6,IDLE_FIT,71.000000,75.000000,J1,4.0,4.0
7,M7,IDLE_FIT,79.000000,83.000000,J3,4.0,4.0
8,M3,ADVANCE,90.000000,95.000000,J5,3.0,5.0
9,M2,IDLE_FIT,100.131177,105.131177,J9,3.0,5.0


## ε 민감도 분석 (PM 빈도 vs Makespan 트레이드오프)